This notebook is responsible for extracting the plain-text of the TLG corpus and filtering texts according to various little filters, **mostly to exclude poetry, fragments and uncertain texts**.

In [1]:
# Documenting
from typing import Generator

# OS
import glob
import os.path

# Data
import csv
import lxml.etree as ET
import pandas
from collections import Counter

# Operations
import regex as re
import unicodedata

# UI
import tqdm

## Constants

In [ ]:
PATH = "patres/*-grc*.xml"  # First1KGreek (tlg*) + PTA (pta*): Greek-language files only
MIN_SIZE = 1000 # Minimum token count
ForbiddenTGs = {
    "tlg0527", "tlg0031"
}

## XML parsing

In [3]:
def normalize(text: str) -> str:
    return unicodedata.normalize("NFKC", re.sub(r"\s{2,}|\n+", " ", text))

cnt = re.compile("\w+")

NS = {"namespaces": {"t": "http://www.tei-c.org/ns/1.0"}}

XSL = """<xsl:stylesheet xmlns:xsl="http://www.w3.org/1999/XSL/Transform"
    xmlns:xs="http://www.w3.org/2001/XMLSchema"
    xmlns:t="http://www.tei-c.org/ns/1.0"
    exclude-result-prefixes="xs"
    version="1.0">
    <xsl:output method="text"/>
    <xsl:template match="t:TEI">
        <xsl:apply-templates select="//t:body" />
    </xsl:template>
    <xsl:template match="t:body">
        <xsl:apply-templates/>
    </xsl:template>
    <xsl:template match="t:label"/>
    <xsl:template match="t:note"/>
    <xsl:template match="t:head"/>
</xsl:stylesheet>"""

XSL = ET.XSLT(ET.fromstring(XSL))

<>:4: SyntaxWarning: invalid escape sequence '\w'
<>:4: SyntaxWarning: invalid escape sequence '\w'
/var/folders/0q/5tjc2wr91qv1h69jgm1s1mzc0000gp/T/ipykernel_7669/220524490.py:4: SyntaxWarning: invalid escape sequence '\w'
  cnt = re.compile("\w+")


## Filtering functions

In [ ]:
def forbidden_author(string) -> bool:
    string = string.lower()
    if "epistula" in string:
        return True
    if "comment" in string:
        return True
    if "scholia" in string:
        return True
    if "lexic" in string:
        return True
    if "acta" in string:
        return True
    if "fragm" in string:
        return True
    if "corpus" in string:
        return True
    if "anonym" in string:
        return True
    if "pseud" in string:
        return True
    if "testam" in string:
        return True
    if "vita" in string:
        return True
    if string.endswith(" et"):
        return True
    if "evangeli" in string:
        return True
    if "floril" in string:
        return True
    if "anthologia" in string:
        return True
    if "paraphras" in string:
        return True
    if "apocal" in string:
        return True
    if "certamen" in string:
        return True
    if "comica adespota" in string:
        return True
    if "commentaria" in string:
        return True
    if "concilia" in string:
        return True
    if "epica adespota" in string:
        return True
    if "etymologicum" in string:
        return True
    if "epistula ecclesiarum" in string:
        return True
    if "gnologium" in string:
        return True
    if "lyrica adespota" in string:
        return True
    if "proverbi" in string:
        return True
    if "oracula" in string:
        return True
    if "periplus" in string:
        return True
    if "incertus" in string:
        return True
    if "socraticorum epistulae" in string:
        return True
    if "socrat" in string:
        return True
    if "[" in string:
        return True
    if "anonym" in string:
        return True
    if "pseudo" in string:
        return True
    if "ps." in string:
        return True
    if "historia" in string:
        return True
    
def is_poetry(xml):
    return len(xml.xpath("//t:l", **NS)) > 0

def has_fragment(xml):
    return len(xml.xpath("//t:div[@type='fragment']", **NS)) > 0

def get_tg(filename):
    return os.path.basename(filename)[:7]   # e.g. tlg4090 or pta0001

def rename_author(author):
    return re.sub(r"(\s+\w+\.)", "", author).replace(" et", "")

## Data wrangling functions

In [5]:
def get_tg(filename):
    return os.path.basename(filename)[:7]

def rename_author(author):
    return re.sub(r"(\s+\w+\.)", "", author).replace(" et", "")

## Data accumulation

In [6]:
# Output Data
data = []

# Count the number of files we have per textgroup
tgs = Counter([
    get_tg(file)
    for file in glob.glob(PATH)
])

# Statistics
passed = 0
tgcount = Counter()
ignored_authors = set()
    
for file in tqdm.tqdm(glob.glob(PATH)):
    try:
        xml = ET.parse(file)
        tg = get_tg(file)
        author = str(xml.xpath("/t:TEI/t:teiHeader/t:fileDesc/t:titleStmt/t:author//text()", **NS)[0])
        title = xml.xpath("/t:TEI/t:teiHeader/t:fileDesc/t:titleStmt/t:title/text()", **NS)[0]
        
        if tg in ForbiddenTGs:
            passed += 1
            continue
        elif forbidden_author(author):
            ignored_authors.add(author)
            continue
        elif is_poetry(xml):
            #print(f"Ignoring {file} for poetry reason")
            ignored_authors.add(author)
            continue
        elif has_fragment(xml):
            #print(f"Ignoring {file} for fragment reason")
            ignored_authors.add(author)
            continue
            
        rawtext = normalize(str(XSL(xml))).strip()
        tokens = cnt.findall(rawtext)
        
        if len(tokens) < MIN_SIZE:
            passed += 1
            continue

        data.append({
            "file": os.path.basename(file)[:-4],
            "orig_author": author,
            "author": rename_author(str(author)),
            "title": str(title),
            "textgroup": tg,
            "tokens": len(tokens),
            "full-text-raw": rawtext
        })
        tgcount[tg] += 1
    except Exception as E:
        passed += 1
        print(f"Failing on {file}: {E}")
        continue
print(passed)
print(ignored_authors)


  0%|          | 0/32 [00:00<?, ?it/s]


 22%|██▏       | 7/32 [00:00<00:00, 54.43it/s]


 41%|████      | 13/32 [00:00<00:00, 35.88it/s]


 69%|██████▉   | 22/32 [00:00<00:00, 41.77it/s]


 84%|████████▍ | 27/32 [00:00<00:00, 40.29it/s]


100%|██████████| 32/32 [00:00<00:00, 43.18it/s]

1
{'Theodoretus', 'Athanasius', 'Eusebius', 'pseudo-Menander', 'Cyril of Alexandria', 'Gregory of Nazianzus'}


## Exporting

In [7]:
df = pandas.DataFrame(data)
print("Before filtering on Title", df.shape)
df = df[~df.title.str.contains("Dub\.|Sp\.|Fragm|Excerpt|(e cod\.)|Suda|recensio|fragm|sp\.|dub\.|(fort\. auctore)|Scholia")]
print("After filtering on Title", df.shape)


df.to_csv("tlg-texts.csv", index=False)
df.head()

<>:3: SyntaxWarning: invalid escape sequence '\.'
<>:3: SyntaxWarning: invalid escape sequence '\.'
/var/folders/0q/5tjc2wr91qv1h69jgm1s1mzc0000gp/T/ipykernel_7669/1736975512.py:3: SyntaxWarning: invalid escape sequence '\.'
  df = df[~df.title.str.contains("Dub\.|Sp\.|Fragm|Excerpt|(e cod\.)|Suda|recensio|fragm|sp\.|dub\.|(fort\. auctore)|Scholia")]
/var/folders/0q/5tjc2wr91qv1h69jgm1s1mzc0000gp/T/ipykernel_7669/1736975512.py:3: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df = df[~df.title.str.contains("Dub\.|Sp\.|Fragm|Excerpt|(e cod\.)|Suda|recensio|fragm|sp\.|dub\.|(fort\. auctore)|Scholia")]


Before filtering on Title (20, 7)
After filtering on Title (19, 7)


,file,orig_author,author,title,textgroup,tokens,full-text-raw
0,tlg2022.tlg008.1st1K-grc1,Gregorius Nazianzenus,Gregorius Nazianzenus,De Theologia (Orat. 28),tlg2022,7358,"Ἐπειδὴ ἀνεκαθήραμεν τῷ λόγῳ τὸν θεόλογον, οἷόν..."
1,tlg2035.tlg132.1st1K-grc1,Athanasius,Athanasius,Oratio III contra Arianos,tlg2035,24206,"1. ΟΙ Ἀρειομανῖται, ὡς ἔοικε, κρίναντες ἅπαξ ἀ..."
2,tlg2018.tlg009.opp-grc1,Eusebius of Caesarea,Eusebius of Caesarea,De Ecclesiastica Theologia (Book 3 only),tlg2018,15504,α ́ ἰὸς διαστρόφως ὁ Μάρκελλος τἀς θείας γραφά...
3,tlg2018.tlg007.1st1K-grc1,Eusebius of Caesarea,Eusebius of Caesarea,των κατά Μαρκέλλου,tlg2018,17886,⟨ πόθεν ὁρμώμενος ὁ Μάρκελλος τουτὶ τὸ σύγγραμ...
4,tlg2022.tlg007.1st1K-grc1,Gregory of Nazianzus,Gregory of Nazianzus,Adversus Eunomianos (orat. 27),tlg2022,2075,Πρὸς τοὺς ἐν λόγῳ κομψοὺς ὁ λόγος. καὶ ἵνα ἀπὸ...


In [8]:
for x in sorted(df.orig_author.unique()):
    print(x)

Athanasius
Athanasius of Alexandria
Eusebius
Eusebius of Caesarea
Gregorius Nazianzenus
Gregory of Nazianzus
Theodoret, Bishop of Cyrus


## Lemmatization

Requires to have set-up `bert-env` virtual environment.

In [9]:
import os
if os.path.exists("./bert-env/bin/python"):
    # Tag First1KGreek texts (slow; skips files already in tagged/)
    !./bert-env/bin/python tag-in-xml.py tlg-texts.csv
else:
    print("bert-env not found — skipping BERT tagging for First1KGreek texts.")
    print("Run install-bert.sh then re-run this cell to tag them.")
    print("PG corpus texts (Step 04b) are pre-tagged and do not need this step.")

2026-07-27 13:06:09,582 SequenceTagger predicts: Dictionary with 1030 tags: <unk>, O, a-p---na-, v2spma---, u--------, d--------, v-papamn-, r--------, l-s---ma-, n-s---ma-, v3siie---, l-s---nn-, l-s---fg-, n-s---fg-, l-s---mg-, n-s---mg-, v3ppia---, i--------, n-s---mn-, v3saia---, p-p---fd-, v-sppamn-, a-s---mn-, n-p---mg-, c--------, v3saoa---, p-s---mn-, l-s---mn-, v3siia---, v-sapamg-, b--------, p-s---cg-, p-s---fd-, l-p---mg-, a-p---mg-, a-s---ma-, v-sppamg-, v3spia---, a-p---ng-, n-p---ng-, _, v3piie---, l-p---md-, a-p---md-, v-pppamn-, p-p---ma-, l-s---fa-, n-s---fa-, n-p---na-, v3paia---



0it [00:00, ?it/s]


0it [00:01, ?it/s]
Traceback (most recent call last):
  File "/Users/chartja/Github/Chryso-Voicu/tag-in-xml.py", line 76, in <module>
    pos_text = get_text_poses(text["full-text-raw"])
  File "/Users/chartja/Github/Chryso-Voicu/tag-in-xml.py", line 63, in get_text_poses
    out.extend(get_poses(sentence))
               ~~~~~~~~~^^^^^^^^^^
  File "/Users/chartja/Github/Chryso-Voicu/tag-in-xml.py", line 55, in get_poses
    tagger.predict(sentence)
    ~~~~~~~~~~~~~~^^^^^^^^^^
  File "/Users/chartja/Github/Chryso-Voicu/bert-env/lib/python3.13/site-packages/flair/models/sequence_tagger_model.py", line 514, in predict
    sentence_tensor, lengths = self._prepare_tensors(batch)
                               ~~~~~~~~~~~~~~~~~~~~~^^^^^^^
  File "/Users/chartja/Github/Chryso-Voicu/bert-env/lib/python3.13/site-packages/flair/models/sequence_tagger_model.py", line 304, in _prepare_tensors
    self.embeddings.embed(sentences)
    ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^
  File "/Users/chartja/Github